# CHIA NER Fine-tuning (BERT)

## Install (stable set)

In [ ]:
!pip -q uninstall -y transformers tokenizers huggingface_hub accelerate datasets pyarrow fsspec requests numpy
!pip -q install -U "numpy==2.0.2"
!pip -q install -U "requests==2.32.4" "fsspec[http]==2025.3.0" "pyarrow==16.1.0"
!pip -q install -U "datasets==4.0.0"
!pip -q install -U --force-reinstall "transformers==4.45.2" "tokenizers==0.20.1" "huggingface_hub==0.26.3" "accelerate==0.32.1"
!pip -q install -U "seqeval==1.2.2"

Runtime → Restart runtime again.

## Imports + version check

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import matthews_corrcoef
import datasets
import transformers
import torch

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

from transformers import Trainer
print("Trainer import OK")

numpy: 2.0.2
pandas: 2.2.2
datasets: 5.0.0
transformers: 5.10.2
torch: 2.11.0+cu128
Trainer import OK


## Mount Google Drive (to save the model)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/thesis_eligibility_ai")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT)

Drive root: /content/drive/MyDrive/thesis_eligibility_ai


## Load CHIA (BigBio)

In [4]:
import requests
from datasets import load_dataset

meta = requests.get("https://datasets-server.huggingface.co/parquet?dataset=bigbio/chia").json()

target = None
for item in meta["parquet_files"]:
    if item.get("config") == "chia_bigbio_kb" and item.get("split") == "train":
        target = item
        break

assert target is not None
url = target["url"]
print("Using parquet:", url)

ds = load_dataset("parquet", data_files={"train": [url]})
chia = ds["train"]
print("Loaded CHIA rows:", len(chia))
print("Example keys:", chia[0].keys())

Using parquet: https://huggingface.co/datasets/bigbio/chia/resolve/refs%2Fconvert%2Fparquet/chia_bigbio_kb/train/0000.parquet


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Loaded CHIA rows: 2000
Example keys: dict_keys(['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'])


## Split datasets

In [5]:
from sklearn.model_selection import GroupKFold

all_rows = list(chia)

def get_document_id(ex, idx):
    return str(ex.get("document_id") or ex.get("id") or f"row_{idx}")

doc_groups = [get_document_id(ex, i) for i, ex in enumerate(all_rows)]

gkf = GroupKFold(n_splits=10)

fold_splits = []
for fold_id, (train_idx, test_idx) in enumerate(gkf.split(all_rows, groups=doc_groups), start=1):
    fold_splits.append({
        "fold": fold_id,
        "train_idx": train_idx.tolist(),
        "test_idx": test_idx.tolist(),
    })

print("Built", len(fold_splits), "document-level folds")

Built 10 document-level folds


Grouped by document_id:

Because NCT00050349_inc and NCT00050349_exc should never land in different folds. That would leak trial-specific wording. This grouping rule is my recommendation for robustness. Li et al say they combined inclusion and exclusion together for each annotated EC file, which points in the same direction.

## Filter CHIA train/dev

In [6]:
from sklearn.model_selection import GroupShuffleSplit

# choose one fold first as pilot
selected_fold = 10 #10 is all
fold_info = fold_splits[selected_fold - 1]

train_idx_full = fold_info["train_idx"]
test_idx = fold_info["test_idx"]

# rows and groups inside the TRAIN portion only
train_rows_full = [all_rows[i] for i in train_idx_full]
train_groups_full = [get_document_id(all_rows[i], i) for i in train_idx_full]

# split train -> train/dev by document_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.1111, random_state=13)
# 0.1111 of the train portion gives about 10% of total as dev
inner_train_pos, dev_pos = next(
    gss.split(train_rows_full, groups=train_groups_full)
)

train_rows = [train_rows_full[i] for i in inner_train_pos]
dev_rows   = [train_rows_full[i] for i in dev_pos]
test_rows  = [all_rows[i] for i in test_idx]

print("Fold", selected_fold)
print("Train rows:", len(train_rows))
print("Dev rows:", len(dev_rows))
print("Test rows:", len(test_rows))

# check: to see no document leakage
def ex_doc_id(ex):
    return str(ex.get("document_id") or ex.get("id"))

train_docs = set(ex_doc_id(ex) for ex in train_rows)
dev_docs   = set(ex_doc_id(ex) for ex in dev_rows)
test_docs  = set(ex_doc_id(ex) for ex in test_rows)

print("Train ∩ Dev:", len(train_docs & dev_docs))
print("Train ∩ Test:", len(train_docs & test_docs))
print("Dev ∩ Test:", len(dev_docs & test_docs))

Fold 10
Train rows: 1600
Dev rows: 200
Test rows: 200
Train ∩ Dev: 0
Train ∩ Test: 0
Dev ∩ Test: 0


Why 0.1111? --> because GroupKFold already gave about:

90% train/10% test

Then you split the 90% train into:

about 80% train/about 10% dev

keeping the original 10% test

Since dev is taken from the 90% block, you want:

0.1/0.9≈0.1111

In [7]:
print("Unique document_ids:", len(set(doc_groups)))

Unique document_ids: 1000


## Extract text + entities (char offsets)

In [8]:
from typing import Dict, Any, List, Tuple

DROP_LABELS = {"Scope"}   # Li used Without Scopes

def get_text_from_example(ex: Dict[str, Any]) -> str:
    passages = ex.get("passages", [])
    if not passages:
        return ""

    p0 = passages[0]
    text = p0.get("text", "")
    if isinstance(text, list):
        text = " ".join([t for t in text if t is not None])

    # NOT strip; keep offsets aligned
    return text or ""


def extract_li_style_flat_entities(ex: Dict[str, Any]) -> Tuple[str, List[Dict[str, Any]]]:
    text = get_text_from_example(ex)
    if not text:
        return "", []

    raw_ents = []

    for ent in ex.get("entities", []):
        label = ent.get("type", "Other")
        if label in DROP_LABELS:
            continue

        offsets = ent.get("offsets", [])
        valid_spans = []
        for span in offsets:
            if not span or len(span) != 2:
                continue
            s, e = int(span[0]), int(span[1])
            if s < 0 or e <= s or e > len(text):
                continue
            valid_spans.append((s, e))

        if not valid_spans:
            continue

        # Li paper disjoint merge -> one continuous entity
        start = min(s for s, _ in valid_spans)
        end   = max(e for _, e in valid_spans)

        raw_ents.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end]
        })

    # sort by start asc, end desc so outer spans come first
    raw_ents.sort(key=lambda x: (x["start"], -(x["end"] - x["start"])))

    flat_ents = []
    for ent in raw_ents:
        s, e = ent["start"], ent["end"]

        # drop if nested inside an already kept entity
        nested = False
        overlap = False

        for kept in flat_ents:
            ks, ke = kept["start"], kept["end"]

            if ks <= s and e <= ke:
                nested = True
                break

            # overlapping but neither contains the other
            if not (e <= ks or s >= ke):
                overlap = True
                break

        if nested:
            continue

        # conservative: skip unresolved overlap conflicts
        if overlap:
            continue

        flat_ents.append(ent)

    return text, flat_ents

## Build BIO labels

In [9]:
def collect_labels_from_flat(rows):
    labels = set()
    for ex in rows:
        _, ents = extract_li_style_flat_entities(ex)
        for ent in ents:
            labels.add(ent["label"])
    return sorted(labels)

all_flat_labels = collect_labels_from_flat(all_rows)
print("Flat labels:", all_flat_labels)

label_list = ["O"]
for lab in all_flat_labels:
    label_list += [f"B-{lab}", f"I-{lab}"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print("Num flat entity types:", len(all_flat_labels))
print("Total BIO tags:", len(label_list))

Flat labels: ['Condition', 'Device', 'Drug', 'Measurement', 'Mood', 'Multiplier', 'Negation', 'Observation', 'Person', 'Procedure', 'Qualifier', 'Reference_point', 'Temporal', 'Value', 'Visit']
Num flat entity types: 15
Total BIO tags: 31


## Create HF datasets

In [10]:
from datasets import Dataset

train_ds = Dataset.from_list(train_rows)
dev_ds   = Dataset.from_list(dev_rows)
test_ds  = Dataset.from_list(test_rows)

print(train_ds)
print(dev_ds)
print(test_ds)

Dataset({
    features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
    num_rows: 1600
})
Dataset({
    features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
    num_rows: 200
})
Dataset({
    features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
    num_rows: 200
})


## Tokenize + align

In [11]:
from transformers import AutoTokenizer

def encode_and_align_flat(example, tokenizer, label2id, max_length=256):
    text, ents = extract_li_style_flat_entities(example)

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True
    )

    offsets = enc["offset_mapping"]
    labels = []

    for (os_, oe_) in offsets:
        # special tokens
        if os_ == oe_ == 0:
            labels.append(-100)
            continue

        assigned = "O"
        for ent in ents:
            s, e, lab = ent["start"], ent["end"], ent["label"]

            if not (oe_ <= s or os_ >= e):   # overlap
                if os_ == s or (os_ >= s and (os_ - s) <= 1):
                    assigned = f"B-{lab}"
                else:
                    assigned = f"I-{lab}"
                break

        labels.append(label2id[assigned])

    enc["labels"] = labels
    enc.pop("offset_mapping")
    return enc

## Train function

In [14]:
%pip uninstall -y torchvision
%pip install -q --upgrade datasets
%pip install -q seqeval

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Runtime → Restart session

In [12]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import json
from pathlib import Path
from transformers import DataCollatorForTokenClassification

def evaluate_seqeval(model, dataloader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)

            for pred_seq, true_seq in zip(preds, labels):
                tl, tp = [], []
                for p_id, t_id in zip(pred_seq, true_seq):
                    if int(t_id.item()) == -100:
                        continue
                    tl.append(id2label[int(t_id.item())])
                    tp.append(id2label[int(p_id.item())])
                all_true.append(tl)
                all_pred.append(tp)

    return {
        "precision": precision_score(all_true, all_pred),
        "recall": recall_score(all_true, all_pred),
        "f1": f1_score(all_true, all_pred),
        "accuracy": accuracy_score(all_true, all_pred),
    }

def train_one_model(model_name: str, out_dir: Path, run_name: str, seed: int = 13):
    torch.manual_seed(seed)
    np.random.seed(seed)

    learning_rate = 5e-5
    batch_size = 8
    max_length = 256
    num_epochs = 10

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # tokenize
    train_tok = train_ds.map(
        lambda ex: encode_and_align_flat(ex, tokenizer, label2id, max_length=max_length),
        batched=False
    )
    dev_tok = dev_ds.map(
        lambda ex: encode_and_align_flat(ex, tokenizer, label2id, max_length=max_length),
        batched=False
    )
    test_tok = test_ds.map(
        lambda ex: encode_and_align_flat(ex, tokenizer, label2id, max_length=max_length),
        batched=False
    )

    # torch format
    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    dev_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    data_collator = DataCollatorForTokenClassification(tokenizer)

    train_loader = DataLoader(train_tok, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
    dev_loader = DataLoader(dev_tok, batch_size=batch_size, shuffle=False, collate_fn=data_collator)
    test_loader = DataLoader(test_tok, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    num_training_steps = num_epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_training_steps),
        num_training_steps=num_training_steps
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_f1 = -1.0
    best_state = None

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

            if step % 50 == 0:
                print(f"epoch {epoch} step {step}/{len(train_loader)} loss={loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} train loss: {avg_loss:.4f}")

        dev_metrics = evaluate_seqeval(model, dev_loader, device)
        print(f"Epoch {epoch} dev:", dev_metrics)

        if dev_metrics["f1"] > best_f1:
            best_f1 = dev_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_seqeval(model, test_loader, device)
    print("Final test:", test_metrics)

    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    meta = {
        "base_model": model_name,
        "run_name": run_name,
        "seed": seed,
        "metrics_dev_best_f1": best_f1,
        "metrics_test": test_metrics,
        "label_list": label_list,
        "dataset": {"name": "bigbio/chia", "config": "chia_bigbio_kb"},
        "fold": selected_fold,
        "epochs": num_epochs,
        "lr": learning_rate,
        "batch_size": batch_size,
        "max_length": max_length,
        "drop_labels": sorted(list(DROP_LABELS)),
    }
    (out_dir / "run_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("Saved to:", out_dir)
    return {
        "best_dev_f1": best_f1,
        "test_metrics": test_metrics,
    }

## Train BioBERT

In [14]:
biobert_name = "dmis-lab/biobert-base-cased-v1.2"
biobert_out = DRIVE_ROOT / "models" / f"biobert_chia_ner_li_fold{selected_fold}_v1"

biobert_metrics = train_one_model(
    model_name=biobert_name,
    out_dir=biobert_out,
    run_name="biobert_chia_ner_v1",
    seed=13
)
print(biobert_metrics)

Device: cuda


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=31` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

epoch 1 step 50/200 loss=2.0720
epoch 1 step 100/200 loss=1.0596
epoch 1 step 150/200 loss=0.7943
epoch 1 step 200/200 loss=0.8530
Epoch 1 train loss: 1.5901
Epoch 1 dev: {'precision': np.float64(0.5238338006627581), 'recall': np.float64(0.5466879489225858), 'f1': np.float64(0.5350169226763863), 'accuracy': 0.7566973015252249}
epoch 2 step 50/200 loss=1.0315
epoch 2 step 100/200 loss=0.6759
epoch 2 step 150/200 loss=0.6073
epoch 2 step 200/200 loss=0.5436
Epoch 2 train loss: 0.7537
Epoch 2 dev: {'precision': np.float64(0.5348889387480368), 'recall': np.float64(0.6342112263899974), 'f1': np.float64(0.5803310613437196), 'accuracy': 0.769114196323817}
epoch 3 step 50/200 loss=0.4945
epoch 3 step 100/200 loss=0.5343
epoch 3 step 150/200 loss=0.3796
epoch 3 step 200/200 loss=0.6142
Epoch 3 train loss: 0.5646
Epoch 3 dev: {'precision': np.float64(0.5530751708428246), 'recall': np.float64(0.6459164671455174), 'f1': np.float64(0.595901337587434), 'accuracy': 0.7764958936253422}
epoch 4 step 50

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/thesis_eligibility_ai/models/biobert_chia_ner_li_fold10_v1
{'best_dev_f1': np.float64(0.6065614208189442), 'test_metrics': {'precision': np.float64(0.6091109478324761), 'recall': np.float64(0.667292728736249), 'f1': np.float64(0.6368758002560818), 'accuracy': 0.7882607319902963}}


## Train PubMedBERT

In [31]:
pubmedbert_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
pubmedbert_out = DRIVE_ROOT / "models" / f"pubmedbert_chia_ner_li_fold{selected_fold}_v1"

pubmedbert_metrics = train_one_model(
    model_name=pubmedbert_name,
    out_dir=pubmedbert_out,
    run_name="pubmedbert_chia_ner_v1",
    seed=13
)
print(pubmedbert_metrics)

Device: cuda


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING

epoch 1 step 50/200 loss=2.3875
epoch 1 step 100/200 loss=1.3949
epoch 1 step 150/200 loss=0.9393
epoch 1 step 200/200 loss=0.8318
Epoch 1 train loss: 1.7693
Epoch 1 dev: {'precision': np.float64(0.5487066593285636), 'recall': np.float64(0.5687393040501997), 'f1': np.float64(0.5585434173669468), 'accuracy': 0.7649874496526764}
epoch 2 step 50/200 loss=1.0150
epoch 2 step 100/200 loss=0.7381
epoch 2 step 150/200 loss=0.6475
epoch 2 step 200/200 loss=0.5815
Epoch 2 train loss: 0.7764
Epoch 2 dev: {'precision': np.float64(0.5335892514395394), 'recall': np.float64(0.6343411294922989), 'f1': np.float64(0.5796194943966642), 'accuracy': 0.7687233669955053}
epoch 3 step 50/200 loss=0.6581
epoch 3 step 100/200 loss=0.5748
epoch 3 step 150/200 loss=0.4936
epoch 3 step 200/200 loss=0.6465
Epoch 3 train loss: 0.6194
Epoch 3 dev: {'precision': np.float64(0.5759030058095479), 'recall': np.float64(0.6503137478608101), 'f1': np.float64(0.6108506363027462), 'accuracy': 0.7875780748350942}
epoch 4 step 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/thesis_eligibility_ai/models/pubmedbert_chia_ner_li_fold10_v1
{'best_dev_f1': np.float64(0.6166710910538891), 'test_metrics': {'precision': np.float64(0.6132698916204071), 'recall': np.float64(0.6687806284231768), 'f1': np.float64(0.6398234969663541), 'accuracy': 0.789153605015674}}


## Train BioClinicalBERT

In [17]:
bioclinical_name = "emilyalsentzer/Bio_ClinicalBERT"
bioclinical_out  = DRIVE_ROOT / "models" / f"bioclinicalbert_chia_ner_li_fold{selected_fold}_v1"

bioclinical_metrics = train_one_model(
    model_name=bioclinical_name,
    out_dir=bioclinical_out,
    run_name="bioclinicalbert_chia_ner_v1",
    seed=13
)
print("BioClinicalBERT:", bioclinical_metrics)

Device: cuda


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

/tmp/ipykernel_3642/1786502792.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_3642/1786502792.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_3642/1786502792.py:115: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1 step 50/200 loss=2.2694
epoch 1 step 100/200 loss=1.3615
epoch 1 step 150/200 loss=0.9281
epoch 1 step 200/200 loss=0.9284
Epoch 1 train loss: 1.7495
Epoch 1 dev: {'precision': np.float64(0.5027279812938426), 'recall': np.float64(0.5147645650438947), 'f1': np.float64(0.5086750788643535), 'accuracy': 0.7553285099726241}
epoch 2 step 50/200 loss=1.0446
epoch 2 step 100/200 loss=0.7102
epoch 2 step 150/200 loss=0.6469
epoch 2 step 200/200 loss=0.5587
Epoch 2 train loss: 0.7877
Epoch 2 dev: {'precision': np.float64(0.509075379030536), 'recall': np.float64(0.6342112263899974), 'f1': np.float64(0.5647950722577588), 'accuracy': 0.7662788423934298}
epoch 3 step 50/200 loss=0.6225
epoch 3 step 100/200 loss=0.4666
epoch 3 step 150/200 loss=0.4122
epoch 3 step 200/200 loss=0.5925
Epoch 3 train loss: 0.5784
Epoch 3 dev: {'precision': np.float64(0.5322041453086694), 'recall': np.float64(0.6352753391859537), 'f1': np.float64(0.5791899102595197), 'accuracy': 0.7754692999608916}
epoch 4 step 5

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/thesis_eligibility_ai/models/bioclinicalbert_chia_ner_li_fold10_v1
BioClinicalBERT: {'best_dev_f1': np.float64(0.6016121152906693), 'test_metrics': {'precision': np.float64(0.5828775267538644), 'recall': np.float64(0.6576334853769789), 'f1': np.float64(0.6180030257186082), 'accuracy': 0.7816158633055584}}


## SciBERT

In [18]:
scibert_name = "allenai/scibert_scivocab_uncased"
scibert_out  = DRIVE_ROOT / "models" / f"scibert_chia_ner_li_fold{selected_fold}_v1"

scibert_metrics = train_one_model(
    model_name=scibert_name,
    out_dir=scibert_out,
    run_name="scibert_chia_ner_v1",
    seed=13
)
print("SciBERT:", scibert_metrics)

Device: cuda


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

/tmp/ipykernel_3642/1786502792.py:115: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1 step 50/200 loss=2.1158
epoch 1 step 100/200 loss=1.0189
epoch 1 step 150/200 loss=0.7779
epoch 1 step 200/200 loss=0.8217
Epoch 1 train loss: 1.5784
Epoch 1 dev: {'precision': np.float64(0.5319553984226272), 'recall': np.float64(0.5599770970512453), 'f1': np.float64(0.5456066945606695), 'accuracy': 0.7579455164585698}
epoch 2 step 50/200 loss=0.9420
epoch 2 step 100/200 loss=0.7300
epoch 2 step 150/200 loss=0.6098
epoch 2 step 200/200 loss=0.5164
Epoch 2 train loss: 0.7389
Epoch 2 dev: {'precision': np.float64(0.5479651162790697), 'recall': np.float64(0.6475808760377899), 'f1': np.float64(0.5936228841359402), 'accuracy': 0.7757094211123723}
epoch 3 step 50/200 loss=0.5300
epoch 3 step 100/200 loss=0.5140
epoch 3 step 150/200 loss=0.4397
epoch 3 step 200/200 loss=0.6017
Epoch 3 train loss: 0.5488
Epoch 3 dev: {'precision': np.float64(0.5607407407407408), 'recall': np.float64(0.6501574577726882), 'f1': np.float64(0.6021476865968447), 'accuracy': 0.776844494892168}
epoch 4 step 5

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/thesis_eligibility_ai/models/scibert_chia_ner_li_fold10_v1
SciBERT: {'best_dev_f1': np.float64(0.6156741497513106), 'test_metrics': {'precision': np.float64(0.5918313570487483), 'recall': np.float64(0.6487579433853264), 'f1': np.float64(0.6189885627669836), 'accuracy': 0.7826272738371384}}


## Comparative table

In [13]:
from pathlib import Path
import json
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)
from sklearn.metrics import matthews_corrcoef


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive/thesis_eligibility_ai")
MODELS_DIR = DRIVE_ROOT / "models"

BATCH_SIZE = 8
MAX_LENGTH = 256

# Works whether your notebook uses selected_fold or fold_id
if "selected_fold" in globals():
    FOLD_ID = selected_fold
elif "fold_id" in globals():
    FOLD_ID = fold_id
else:
    raise NameError("Neither selected_fold nor fold_id is defined.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Fold:", FOLD_ID)


# ------------------------------------------------------------
# Model display names
# ------------------------------------------------------------
def short_name(folder_name: str) -> str:
    name = folder_name.lower()

    if "pubmedbert" in name:
        return "PubMedBERT"
    if "bioclinicalbert" in name:
        return "BioClinicalBERT"
    if "scibert" in name:
        return "SciBERT"
    if "biobert" in name:
        return "BioBERT"

    return folder_name


# ------------------------------------------------------------
# Calculate multiclass token-level MCC
# ------------------------------------------------------------
def calculate_test_mcc(model, dataloader, device):
    model.eval()

    true_ids = []
    predicted_ids = []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch.pop("labels").to(device)
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)

            valid_mask = labels != -100

            true_ids.extend(
                labels[valid_mask].detach().cpu().tolist()
            )
            predicted_ids.extend(
                predictions[valid_mask].detach().cpu().tolist()
            )

    return float(matthews_corrcoef(true_ids, predicted_ids))


# ------------------------------------------------------------
# Read saved metrics and calculate MCC for each trained model
# ------------------------------------------------------------
rows = []

for model_dir in sorted(MODELS_DIR.iterdir()):
    if not model_dir.is_dir():
        continue

    if f"chia_ner_li_fold{FOLD_ID}_v1" not in model_dir.name:
        continue

    meta_path = model_dir / "run_metadata.json"

    if not meta_path.exists():
        continue

    print("\nEvaluating MCC for:", short_name(model_dir.name))

    metadata = json.loads(
        meta_path.read_text(encoding="utf-8")
    )

    test_metrics = metadata.get("metrics_test", {})

    # Load the just-trained saved model
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        use_fast=True,
    )

    model = AutoModelForTokenClassification.from_pretrained(
        model_dir
    ).to(device)

    # Tokenize the same held-out test set
    test_tok = test_ds.map(
        lambda example: encode_and_align_flat(
            example,
            tokenizer,
            label2id,
            max_length=MAX_LENGTH,
        ),
        batched=False,
        remove_columns=test_ds.column_names,
    )

    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer
    )

    test_loader = DataLoader(
        test_tok,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator,
    )

    # Calculate MCC
    test_mcc = calculate_test_mcc(
        model=model,
        dataloader=test_loader,
        device=device,
    )

    print("MCC:", round(test_mcc, 4))

    # Save MCC together with the existing test metrics
    test_metrics["mcc"] = test_mcc
    metadata["metrics_test"] = test_metrics

    meta_path.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    rows.append({
        "model": short_name(model_dir.name),
        "dev_f1": metadata.get("metrics_dev_best_f1"),
        "test_precision": test_metrics.get("precision"),
        "test_recall": test_metrics.get("recall"),
        "test_f1": test_metrics.get("f1"),
        "test_accuracy": test_metrics.get("accuracy"),
        "test_mcc": test_mcc,
    })

    del model
    del tokenizer
    del test_loader
    del test_tok

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Final comparison table
# ------------------------------------------------------------
df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError(
        f"No saved models were found for fold {FOLD_ID} "
        f"in {MODELS_DIR}"
    )

percentage_columns = [
    "dev_f1",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_accuracy",
]

for column in percentage_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )
    df[column] = (df[column] * 100).round(2)

# MCC remains between -1 and 1
df["test_mcc"] = pd.to_numeric(
    df["test_mcc"],
    errors="coerce",
).round(4)

df = (
    df.sort_values(
        by="test_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")
print(df.to_string(index=False))




Device: cuda
Fold: 10

Evaluating MCC for: BioBERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

MCC: 0.7457

Evaluating MCC for: BioClinicalBERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

MCC: 0.7322

Evaluating MCC for: SciBERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

MCC: 0.7267

Final model comparison:
          model  dev_f1  test_precision  test_recall  test_f1  test_accuracy  test_mcc
     PubMedBERT   61.67           61.33        66.88    63.98          78.92    0.7322
        BioBERT   60.66           60.91        66.73    63.69          78.83    0.7457
        SciBERT   61.57           59.18        64.88    61.90          78.26    0.7267
BioClinicalBERT   60.16           58.29        65.76    61.80          78.16    0.7381


In [14]:
df.style.background_gradient(
    subset=["dev_f1", "test_precision", "test_recall", "test_f1", "test_accuracy", "test_mcc"],
    cmap="Blues"
)

,model,dev_f1,test_precision,test_recall,test_f1,test_accuracy,test_mcc
0,PubMedBERT,61.670000,61.330000,66.880000,63.980000,78.920000,0.732200
1,BioBERT,60.660000,60.910000,66.730000,63.690000,78.830000,0.745700
2,SciBERT,61.570000,59.180000,64.880000,61.900000,78.260000,0.726700
3,BioClinicalBERT,60.160000,58.290000,65.760000,61.800000,78.160000,0.738100


# Full training

## Load full CHIA from BigBio

In [22]:
meta = requests.get("https://datasets-server.huggingface.co/parquet?dataset=bigbio/chia").json()

target = None
for item in meta["parquet_files"]:
    if item.get("config") == "chia_bigbio_kb" and item.get("split") == "train":
        target = item
        break

assert target is not None, "Could not find CHIA parquet file."

url = target["url"]
print("Using parquet:", url)

ds = load_dataset("parquet", data_files={"train": [url]})
chia = ds["train"]

print("Loaded CHIA rows:", len(chia))
print("Example keys:", chia[0].keys())

Using parquet: https://huggingface.co/datasets/bigbio/chia/resolve/refs%2Fconvert%2Fparquet/chia_bigbio_kb/train/0000.parquet
Loaded CHIA rows: 2000
Example keys: dict_keys(['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'])


## Li-style preprocessing: flatten CHIA for token classification

In [23]:
DROP_LABELS = {"Scope"}   # Li-style: Without Scopes

def get_text_from_example(ex: Dict[str, Any]) -> str:
    passages = ex.get("passages", [])
    if not passages:
        return ""

    p0 = passages[0]
    text = p0.get("text", "")
    if isinstance(text, list):
        text = " ".join([t for t in text if t is not None])

    # keep original offsets aligned
    return text or ""


def extract_li_style_flat_entities(ex: Dict[str, Any]) -> Tuple[str, List[Dict[str, Any]]]:
    text = get_text_from_example(ex)
    if not text:
        return "", []

    raw_ents = []

    for ent in ex.get("entities", []):
        label = ent.get("type", "Other")
        if label in DROP_LABELS:
            continue

        offsets = ent.get("offsets", [])
        valid_spans = []

        for span in offsets:
            if not span or len(span) != 2:
                continue
            s, e = int(span[0]), int(span[1])
            if s < 0 or e <= s or e > len(text):
                continue
            valid_spans.append((s, e))

        if not valid_spans:
            continue

        # merge disjoint spans into one continuous span
        start = min(s for s, _ in valid_spans)
        end   = max(e for _, e in valid_spans)

        raw_ents.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end]
        })

    # outer spans first
    raw_ents.sort(key=lambda x: (x["start"], -(x["end"] - x["start"])))

    flat_ents = []
    for ent in raw_ents:
        s, e = ent["start"], ent["end"]

        nested = False
        overlap = False

        for kept in flat_ents:
            ks, ke = kept["start"], kept["end"]

            # nested inside a kept span
            if ks <= s and e <= ke:
                nested = True
                break

            # unresolved overlap
            if not (e <= ks or s >= ke):
                overlap = True
                break

        if nested:
            continue
        if overlap:
            continue

        flat_ents.append(ent)

    return text, flat_ents

In [24]:
all_rows = list(chia)

text0, ents0 = extract_li_style_flat_entities(all_rows[0])
print("Text length:", len(text0))
print("Num flat ents:", len(ents0))
print("First 10 ents:", ents0[:10])

Text length: 1564
Num flat ents: 55
First 10 ents: [{'start': 14, 'end': 25, 'label': 'Qualifier', 'text': 'symptomatic'}, {'start': 26, 'end': 40, 'label': 'Condition', 'text': 'CNS metastases'}, {'start': 44, 'end': 70, 'label': 'Condition', 'text': 'leptomeningeal involvement'}, {'start': 92, 'end': 108, 'label': 'Condition', 'text': 'brain metastases'}, {'start': 110, 'end': 116, 'label': 'Negation', 'text': 'unless'}, {'start': 144, 'end': 151, 'label': 'Procedure', 'text': 'treated'}, {'start': 164, 'end': 179, 'label': 'Qualifier', 'text': 'been stable for'}, {'start': 180, 'end': 220, 'label': 'Temporal', 'text': 'at least six months prior to study start'}, {'start': 238, 'end': 248, 'label': 'Observation', 'text': 'history of'}, {'start': 249, 'end': 265, 'label': 'Condition', 'text': 'brain metastases'}]


## Build full label set from all flattened CHIA

In [25]:
def collect_labels_from_flat(rows):
    labels = set()
    for ex in rows:
        _, ents = extract_li_style_flat_entities(ex)
        for ent in ents:
            labels.add(ent["label"])
    return sorted(labels)

all_flat_labels = collect_labels_from_flat(all_rows)
print("Flat labels:", all_flat_labels)

label_list = ["O"]
for lab in all_flat_labels:
    label_list += [f"B-{lab}", f"I-{lab}"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print("Num flat entity types:", len(all_flat_labels))
print("Total BIO tags:", len(label_list))

Flat labels: ['Condition', 'Device', 'Drug', 'Measurement', 'Mood', 'Multiplier', 'Negation', 'Observation', 'Person', 'Procedure', 'Qualifier', 'Reference_point', 'Temporal', 'Value', 'Visit']
Num flat entity types: 15
Total BIO tags: 31


## Create one full dataset

In [26]:
full_ds = Dataset.from_list(all_rows)
print(full_ds)

Dataset({
    features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
    num_rows: 2000
})


## Tokenize and align labels

In [27]:
def encode_and_align_flat(example, tokenizer, label2id, max_length=256):
    text, ents = extract_li_style_flat_entities(example)

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True
    )

    offsets = enc["offset_mapping"]
    labels = []

    for (os_, oe_) in offsets:
        # special tokens
        if os_ == oe_ == 0:
            labels.append(-100)
            continue

        assigned = "O"
        for ent in ents:
            s, e, lab = ent["start"], ent["end"], ent["label"]

            if not (oe_ <= s or os_ >= e):
                if os_ == s or (os_ >= s and (os_ - s) <= 1):
                    assigned = f"B-{lab}"
                else:
                    assigned = f"I-{lab}"
                break

        labels.append(label2id[assigned])

    enc["labels"] = labels
    enc.pop("offset_mapping")
    return enc

## Training function for the final full-data model

In [28]:
def train_full_model(model_name: str, out_dir: Path, run_name: str, seed: int = 13):
    torch.manual_seed(seed)
    np.random.seed(seed)

    learning_rate = 5e-5
    batch_size = 8
    max_length = 256
    num_epochs = 10

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    full_tok = full_ds.map(
        lambda ex: encode_and_align_flat(ex, tokenizer, label2id, max_length=max_length),
        batched=False
    )

    full_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    data_collator = DataCollatorForTokenClassification(tokenizer)
    full_loader = DataLoader(full_tok, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    num_training_steps = num_epochs * len(full_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_training_steps),
        num_training_steps=num_training_steps
    )

    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    train_losses = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0

        for step, batch in enumerate(full_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

            if step % 50 == 0:
                print(f"epoch {epoch} step {step}/{len(full_loader)} loss={loss.item():.4f}")

        avg_loss = total_loss / len(full_loader)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch} full-train loss: {avg_loss:.4f}")

    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    meta = {
        "base_model": model_name,
        "run_name": run_name,
        "training_mode": "full_data_final_model",
        "dataset": {"name": "bigbio/chia", "config": "chia_bigbio_kb"},
        "drop_labels": sorted(list(DROP_LABELS)),
        "num_rows": len(all_rows),
        "num_flat_labels": len(all_flat_labels),
        "label_list": label_list,
        "epochs": num_epochs,
        "lr": learning_rate,
        "batch_size": batch_size,
        "max_length": max_length,
        "seed": seed,
        "train_losses": train_losses,
    }
    (out_dir / "run_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("Saved full-data model to:", out_dir)

## Train the final selected model: PubMedBERT

In [29]:
pubmedbert_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
pubmedbert_out  = MODELS_DIR / "pubmedbert_chia_ner_li_full_v1"

train_full_model(
    model_name=pubmedbert_name,
    out_dir=pubmedbert_out,
    run_name="pubmedbert_chia_ner_li_full_v1",
    seed=13
)

Device: cuda


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING

epoch 1 step 50/250 loss=2.6544
epoch 1 step 100/250 loss=1.7000
epoch 1 step 150/250 loss=1.1302
epoch 1 step 200/250 loss=0.8439
epoch 1 step 250/250 loss=0.6473
Epoch 1 full-train loss: 1.6808
epoch 2 step 50/250 loss=1.4261
epoch 2 step 100/250 loss=0.5479
epoch 2 step 150/250 loss=0.6387
epoch 2 step 200/250 loss=0.7892
epoch 2 step 250/250 loss=1.1350
Epoch 2 full-train loss: 0.7973
epoch 3 step 50/250 loss=0.6047
epoch 3 step 100/250 loss=0.6708
epoch 3 step 150/250 loss=0.4715
epoch 3 step 200/250 loss=0.5018
epoch 3 step 250/250 loss=0.7799
Epoch 3 full-train loss: 0.6253
epoch 4 step 50/250 loss=0.5255
epoch 4 step 100/250 loss=0.4195
epoch 4 step 150/250 loss=0.6080
epoch 4 step 200/250 loss=0.3999
epoch 4 step 250/250 loss=0.5163
Epoch 4 full-train loss: 0.5161
epoch 5 step 50/250 loss=0.3893
epoch 5 step 100/250 loss=0.4141
epoch 5 step 150/250 loss=0.5388
epoch 5 step 200/250 loss=0.4476
epoch 5 step 250/250 loss=0.4140
Epoch 5 full-train loss: 0.4476
epoch 6 step 50/250 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved full-data model to: /content/drive/MyDrive/thesis_eligibility_ai/models/pubmedbert_chia_ner_li_full_v1


## train BioBERT full-data model

In [30]:
biobert_name = "dmis-lab/biobert-base-cased-v1.2"
biobert_out  = MODELS_DIR / "biobert_chia_ner_li_full_v1"

train_full_model(
    model_name=biobert_name,
    out_dir=biobert_out,
    run_name="biobert_chia_ner_li_full_v1",
    seed=13
)

Device: cuda


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=31` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

epoch 1 step 50/250 loss=2.3579
epoch 1 step 100/250 loss=1.3363
epoch 1 step 150/250 loss=1.0141
epoch 1 step 200/250 loss=0.7398
epoch 1 step 250/250 loss=0.6892
Epoch 1 full-train loss: 1.5107
epoch 2 step 50/250 loss=1.3452
epoch 2 step 100/250 loss=0.6539
epoch 2 step 150/250 loss=0.7576
epoch 2 step 200/250 loss=0.8864
epoch 2 step 250/250 loss=0.9329
Epoch 2 full-train loss: 0.7370
epoch 3 step 50/250 loss=0.5698
epoch 3 step 100/250 loss=0.5605
epoch 3 step 150/250 loss=0.4559
epoch 3 step 200/250 loss=0.4777
epoch 3 step 250/250 loss=0.6782
Epoch 3 full-train loss: 0.5516
epoch 4 step 50/250 loss=0.4477
epoch 4 step 100/250 loss=0.3260
epoch 4 step 150/250 loss=0.5860
epoch 4 step 200/250 loss=0.3064
epoch 4 step 250/250 loss=0.4322
Epoch 4 full-train loss: 0.4135
epoch 5 step 50/250 loss=0.2332
epoch 5 step 100/250 loss=0.3485
epoch 5 step 150/250 loss=0.2823
epoch 5 step 200/250 loss=0.2311
epoch 5 step 250/250 loss=0.3658
Epoch 5 full-train loss: 0.3120
epoch 6 step 50/250 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved full-data model to: /content/drive/MyDrive/thesis_eligibility_ai/models/biobert_chia_ner_li_full_v1
